# Classificação de fontes de energia (ANEEL)

Neste notebook treinamos 3 modelos de Machine Learning para descobrir a **fonte de energia** de uma usina a partir da potência (kW), latitude e longitude.

## 1. Importando as bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Algoritmos do sklearn para classificação
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

# Separação dos dados de treino e teste
from sklearn.model_selection import train_test_split

# Métricas para classificação
from sklearn.metrics import accuracy_score         # acurácia
from sklearn.metrics import confusion_matrix       # onde o modelo errou
from sklearn.metrics import classification_report  # resumo de métricas

## 2. Carregando e explorando os dados

In [ ]:
dados_cp = pd.read_csv("https://raw.githubusercontent.com/prof-atritiack/CHECKPOINT_02_SERS_1CC_2SEM/refs/heads/main/aneel_classificacao_orange.csv")
dados_cp.head()

In [ ]:
dados_cp.isnull().sum()  # Há valores ausentes?

In [ ]:
dados_cp.info()  # Tipo de dado de cada coluna

In [ ]:
dados_cp["fonte"].unique()  # Valores únicos da coluna fonte

In [ ]:
dados_cp["fonte"].value_counts()  # Quantidade por fonte (nosso alvo)

In [ ]:
dados_cp.describe()  # Estatísticas gerais

In [ ]:
dados_cp[["potencia_kw","latitude","longitude"]].value_counts()

## 3. Separando entrada (X) e alvo (y)

Usamos `potencia_kw`, `latitude` e `longitude` para prever a `fonte`. O `stratify=y` mantém a proporção de cada fonte no treino e no teste.

In [ ]:
x = dados_cp[["potencia_kw","latitude","longitude"]]  # atributos de entrada
y = dados_cp["fonte"]                                   # o que queremos prever

# train_test_split devolve: x_train, x_test, y_train, y_test (nessa ordem)
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, stratify=y, random_state=42
)

## 4. Treinando os modelos

In [ ]:
modelo_lr = LogisticRegression()          # regressão logística
modelo_lr.fit(x_train, y_train)

modelo_knn = KNeighborsClassifier()       # k vizinhos mais próximos
modelo_knn.fit(x_train, y_train)

modelo_random = RandomForestClassifier()  # floresta aleatória
modelo_random.fit(x_train, y_train)

## 5. Fazendo previsões e medindo a acurácia

In [ ]:
y_knn = modelo_knn.predict(x_test)
y_lr = modelo_lr.predict(x_test)
y_random = modelo_random.predict(x_test)

acuracia_lr = accuracy_score(y_test, y_lr)
acuracia_knn = accuracy_score(y_test, y_knn)
acuracia_random = accuracy_score(y_test, y_random)
acuracia_lr, acuracia_knn, acuracia_random

## 6. Métricas detalhadas

In [ ]:
print("--- Métricas para Regressão Logística ---")
print(classification_report(y_test, y_lr, zero_division=0))
print("Matriz de Confusão para Regressão Logística:")
print(confusion_matrix(y_test, y_lr))

print("\n--- Métricas para K-Nearest Neighbors (KNN) ---")
print(classification_report(y_test, y_knn, zero_division=0))
print("Matriz de Confusão para K-Nearest Neighbors (KNN):")
print(confusion_matrix(y_test, y_knn))

print("\n--- Métricas para Random Forest ---")
print(classification_report(y_test, y_random, zero_division=0))
print("Matriz de Confusão para Random Forest:")
print(confusion_matrix(y_test, y_random))

## 7. Matrizes de confusão em gráfico

In [ ]:
# A matriz de confusão usa as classes em ordem alfabética
classes = sorted(y.unique())

plt.figure(figsize=(18, 5))

plt.subplot(1, 3, 1)
sns.heatmap(confusion_matrix(y_test, y_lr), annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes)
plt.title('Regressão Logística')
plt.xlabel('Previsto')
plt.ylabel('Real')

plt.subplot(1, 3, 2)
sns.heatmap(confusion_matrix(y_test, y_knn), annot=True, fmt='d', cmap='Greens',
            xticklabels=classes, yticklabels=classes)
plt.title('K-Nearest Neighbors (KNN)')
plt.xlabel('Previsto')
plt.ylabel('Real')

plt.subplot(1, 3, 3)
sns.heatmap(confusion_matrix(y_test, y_random), annot=True, fmt='d', cmap='Oranges',
            xticklabels=classes, yticklabels=classes)
plt.title('Random Forest')
plt.xlabel('Previsto')
plt.ylabel('Real')

plt.tight_layout()
plt.show()